#  PhoGPT ABSA Auto-Labeling Tool

**Pure LLM approach** - Không sử dụng keywords, chỉ dùng PhoBERT understanding

---

## ️ Setup

1. **Runtime Settings**: Menu → Runtime → Change runtime type → **GPU (T4)**
2. **Upload file Excel**: Chạy cell "Upload Input File" bên dưới
3. **Configure settings**: Điều chỉnh `LIMIT` và `INPUT_FILENAME`
4. **Run all cells**: Runtime → Run all
5. **Download results**: File sẽ tự động download

---

##  Step 1: Install Dependencies

In [ ]:
!pip install -q torch transformers accelerate bitsandbytes pandas openpyxl tqdm

# Check GPU
import torch
print(f" GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

##  Step 2: Upload Input File

In [ ]:
from google.colab import files
import os

print("Upload your Excel file containing reviews...")
uploaded = files.upload()

# Get filename
INPUT_FILENAME = list(uploaded.keys())[0]
print(f"\n Uploaded: {INPUT_FILENAME}")

## ️ Step 3: Configuration

In [ ]:
# ========== CONFIGURATION ==========

# Limit number of reviews (set to None for all)
LIMIT = 50  # ️ Change this: 50 for testing, None for full dataset

# Use 8-bit quantization (saves memory)
USE_8BIT = True  # Set to False if you have A100 GPU

# Output filename
OUTPUT_FILENAME = "phogpt_labeled_output.xlsx"

# Model settings
MODEL_NAME = "vinai/PhoGPT-7B5-Instruct"
MAX_NEW_TOKENS = 512
TEMPERATURE = 0.1

print(f"Input:  {INPUT_FILENAME}")
print(f"Output: {OUTPUT_FILENAME}")
print(f"Limit:  {LIMIT if LIMIT else 'All reviews'}")
print(f"8-bit:  {USE_8BIT}")

##  Step 4: Load PhoGPT Model

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import pandas as pd
import json
import re
from tqdm import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Loading PhoGPT model: {MODEL_NAME}...")
print(f"This may take 5-10 minutes for first time download...\n")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

# Load model
load_kwargs = {
    "trust_remote_code": True,
    "torch_dtype": torch.float16,
}

if USE_8BIT:
    load_kwargs["load_in_8bit"] = True
    print("Loading in 8-bit mode (saves memory)...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    **load_kwargs
)

if not USE_8BIT:
    model = model.to(DEVICE)

model.eval()

print("\n Model loaded successfully!")

## ️ Step 5: Define Labeling Functions

In [ ]:
# 9 Aspects from annotation guideline
ASPECTS = [
    'Chất lượng sản phẩm',
    'Hiệu năng & Trải nghiệm',
    'Đúng mô tả',
    'Giá cả & Khuyến mãi',
    'Vận chuyển',
    'Đóng gói',
    'Dịch vụ & Thái độ Shop',
    'Bảo hành & Đổi trả',
    'Tính xác thực',
]

def create_prompt(review: str) -> str:
    """Create prompt for PhoGPT."""
    prompt = f"""### Instruction:
Bạn là chuyên gia phân tích cảm xúc cho bình luận thương mại điện tử.
Nhiệm vụ: Phân tích bình luận sau theo 9 khía cạnh và gán nhãn cảm xúc.

**9 Khía cạnh:**
1. Chất lượng sản phẩm - chất liệu, độ bền, form dáng
2. Hiệu năng & Trải nghiệm - trải nghiệm sử dụng, hiệu suất
3. Đúng mô tả - độ chính xác so với hình ảnh/mô tả
4. Giá cả & Khuyến mãi - giá trị, ưu đãi
5. Vận chuyển - tốc độ, chất lượng giao hàng
6. Đóng gói - bao bì, đóng gói sản phẩm
7. Dịch vụ & Thái độ Shop - CSKH, thái độ người bán
8. Bảo hành & Đổi trả - chính sách bảo hành, đổi trả
9. Tính xác thực - hàng thật/giả, nguồn gốc

**Nhãn cảm xúc:**
- 1: Tích cực (khen, hài lòng)
- 0: Trung lập (nhắc đến nhưng không rõ cảm xúc)
- -1: Tiêu cực (chê, không hài lòng)
- 2: Không nhắc đến
- [-1,1]: Đa cực (vừa khen vừa chê cùng khía cạnh)

**Bình luận:**
\"{review}\"

**Yêu cầu:**
- Đọc kỹ toàn bộ bình luận
- Xác định từng khía cạnh được đề cập
- Gán nhãn cảm xúc chính xác
- Chú ý trường hợp đa cực (có từ \"nhưng\", \"mà\")
- Trả về CHÍNH XÁC định dạng JSON bên dưới, KHÔNG giải thích thêm

### Response:
```json
{{
  \"Chất lượng sản phẩm\": <nhãn>,
  \"Hiệu năng & Trải nghiệm\": <nhãn>,
  \"Đúng mô tả\": <nhãn>,
  \"Giá cả & Khuyến mãi\": <nhãn>,
  \"Vận chuyển\": <nhãn>,
  \"Đóng gói\": <nhãn>,
  \"Dịch vụ & Thái độ Shop\": <nhãn>,
  \"Bảo hành & Đổi trả\": <nhãn>,
  \"Tính xác thực\": <nhãn>
}}
```"""
    return prompt

def parse_response(response: str) -> dict:
    """Parse PhoGPT response to extract labels."""
    try:
        json_match = re.search(r'\{[^}]+\}', response, re.DOTALL)
        if json_match:
            json_str = json_match.group(0)
            labels = json.loads(json_str)
            result = {}
            for aspect in ASPECTS:
                result[aspect] = labels.get(aspect, 2)
            return result
    except Exception as e:
        pass
    return {asp: 2 for asp in ASPECTS}

def label_review(review: str) -> dict:
    """Label single review."""
    if not review or not str(review).strip():
        return {asp: 2 for asp in ASPECTS}
    
    prompt = create_prompt(str(review))
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    
    if torch.cuda.is_available():
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            do_sample=True,
            top_p=0.95,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = response[len(prompt):]
    
    return parse_response(response)

print(" Labeling functions defined")

##  Step 6: Run Labeling

In [ ]:
# Load input data
print(f"Loading {INPUT_FILENAME}...")
df = pd.read_excel(INPUT_FILENAME)
print(f"Total reviews: {len(df)}\n")

# Apply limit
if LIMIT:
    df = df.head(LIMIT)
    print(f"Processing first {LIMIT} reviews\n")

# Initialize aspect columns
for aspect in ASPECTS:
    df[aspect] = 2

# Label each review
print(f"Starting labeling...")
print(f"Estimated time: ~{len(df) * 15 / 60:.1f} minutes\n")

for idx in tqdm(range(len(df)), desc="Labeling reviews"):
    review = df.iloc[idx]['reviewContent']
    
    try:
        labels = label_review(review)
        for aspect, label in labels.items():
            df.at[idx, aspect] = label
    except Exception as e:
        print(f"Error at row {idx}: {e}")
        continue

print("\n Labeling complete!")

##  Step 7: Show Statistics

In [ ]:
print("="*60)
print("LABELING STATISTICS")
print("="*60)

for aspect in ASPECTS:
    mentions = (df[aspect] != 2).sum()
    pos = (df[aspect] == 1).sum()
    neg = (df[aspect] == -1).sum()
    neu = (df[aspect] == 0).sum()
    multi = df[aspect].astype(str).str.contains('\[').sum()
    
    print(f"{aspect:30} | Mentions: {mentions:4} | POS: {pos:3} | NEG: {neg:3} | NEU: {neu:3} | Multi: {multi:2}")

print("="*60)

##  Step 8: Save & Download Results

In [ ]:
# Save to Excel
df.to_excel(OUTPUT_FILENAME, index=False)
print(f"\n Saved to: {OUTPUT_FILENAME}")

# Download file
from google.colab import files
print("\nDownloading file...")
files.download(OUTPUT_FILENAME)

print("\n Done! Check your downloads folder.")

---

##  Notes

- **First run:** Model download takes ~5-10 minutes
- **Speed:** ~10-15 seconds per review on T4 GPU
- **Memory:** Uses ~10-12GB VRAM with 8-bit quantization
- **Labels:** 1 (POS), 0 (NEU), -1 (NEG), 2 (NOT_MENTIONED), [-1,1] (MULTI-POLARITY)

##  Troubleshooting

**OOM Error:**
- Set `USE_8BIT = True`
- Reduce `LIMIT` to smaller number

**Slow inference:**
- Check GPU is enabled: Runtime → Change runtime type → GPU
- Verify GPU usage: `!nvidia-smi`

**Parse errors:**
- Check cell output for errors
- Some reviews may get default labels (2)

---